# GNN training diagnostics — 900-epoch study

Источник: `results/full_full/history/gnn_signed_q95_inductive_molecule_fold1.csv`  
Конфигурация: GraphSAGE signed · q95 · inductive_molecule · fold 1 · seed 42

**Ключевые эпохи:**
- best val AUPRC → epoch 245 (downstream probe: AUPRC ~0.667)
- best test AUPRC (диагностика) → epoch 543
- final → epoch 900

Downstream XGBoost probe на 900-эп энкодере: AUROC 0.807 (+0.024 vs 300-эп).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

df = pd.read_csv(ROOT / "results/full_full/history/gnn_signed_q95_inductive_molecule_fold1.csv")
print(f"epochs: {len(df)}  |  columns: {list(df.columns)}")

best_val_auprc_ep  = int(df["val_AUPRC"].idxmax())  + 1
best_test_auprc_ep = int(df["test_AUPRC"].idxmax()) + 1
print(f"Best val  AUPRC = {df['val_AUPRC'].max():.4f}  @ epoch {best_val_auprc_ep}")
print(f"Best test AUPRC = {df['test_AUPRC'].max():.4f}  @ epoch {best_test_auprc_ep}  (diagnostic only)")

In [ ]:
ep = df["epoch"].values

METRICS = [
    ("AUROC", "AUROC"),
    ("AUPRC", "AUPRC"),
    ("MCC",   "MCC"),
    ("F1",    "F1"),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
axes = axes.ravel()

SMOOTH = 15  # rolling window for readability

for ax, (col, label) in zip(axes, METRICS):
    val_raw  = df[f"val_{col}"].values
    test_raw = df[f"test_{col}"].values

    val_s  = pd.Series(val_raw).rolling(SMOOTH, center=True, min_periods=1).mean().values
    test_s = pd.Series(test_raw).rolling(SMOOTH, center=True, min_periods=1).mean().values

    ax.plot(ep, val_raw,  color="steelblue",  alpha=0.25, lw=0.8)
    ax.plot(ep, test_raw, color="tomato",      alpha=0.25, lw=0.8)
    ax.plot(ep, val_s,    color="steelblue",   lw=1.8, label="val")
    ax.plot(ep, test_s,   color="tomato",      lw=1.8, label="test (diag)")

    # mark key epochs
    ax.axvline(best_val_auprc_ep,  ls="--", color="steelblue", alpha=0.7, lw=1.2,
               label=f"best val AUPRC (ep {best_val_auprc_ep})")
    ax.axvline(best_test_auprc_ep, ls=":",  color="tomato",    alpha=0.7, lw=1.2,
               label=f"best test AUPRC (ep {best_test_auprc_ep})")

    ax.set_ylabel(label); ax.set_title(label)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(100))
    ax.grid(alpha=0.3)
    if col == "AUROC":
        ax.legend(fontsize=8, loc="lower right")

for ax in axes[-2:]:
    ax.set_xlabel("epoch")

fig.suptitle("GNN signed q95 · inductive_molecule · fold1 — val vs test dynamics",
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Training loss + val AUPRC on one axis (dual y)
fig, ax1 = plt.subplots(figsize=(13, 4))

loss_s = pd.Series(df["train_loss"].values).rolling(15, center=True, min_periods=1).mean().values
ax1.plot(ep, df["train_loss"].values, color="gray", alpha=0.2, lw=0.8)
ax1.plot(ep, loss_s, color="gray", lw=1.5, label="train loss")
ax1.set_ylabel("train loss", color="gray")
ax1.tick_params(axis="y", labelcolor="gray")

ax2 = ax1.twinx()
val_au = pd.Series(df["val_AUPRC"].values).rolling(15, center=True, min_periods=1).mean().values
ax2.plot(ep, df["val_AUPRC"].values, color="steelblue", alpha=0.2, lw=0.8)
ax2.plot(ep, val_au, color="steelblue", lw=1.8, label="val AUPRC")
ax2.axvline(best_val_auprc_ep, ls="--", color="steelblue", alpha=0.6, lw=1.2)
ax2.set_ylabel("val AUPRC", color="steelblue")
ax2.tick_params(axis="y", labelcolor="steelblue")

ax1.set_xlabel("epoch")
ax1.xaxis.set_major_locator(ticker.MultipleLocator(100))
ax1.grid(alpha=0.3)

lines1, labs1 = ax1.get_legend_handles_labels()
lines2, labs2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labs1 + labs2, loc="upper right", fontsize=9)
plt.title("Train loss vs val AUPRC")
plt.tight_layout()
plt.show()

In [ ]:
# Val / test correlation scatter per metric
from scipy.stats import spearmanr

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, (col, label) in zip(axes, METRICS):
    v = df[f"val_{col}"].values
    t = df[f"test_{col}"].values
    rho, _ = spearmanr(v, t)
    ax.scatter(v, t, s=2, alpha=0.3, c=ep, cmap="viridis")
    ax.set_xlabel(f"val {label}"); ax.set_ylabel(f"test {label}")
    ax.set_title(f"{label}  ρ={rho:.3f}")
    ax.grid(alpha=0.3)

fig.suptitle("Val ↔ Test per-epoch correlation (colour = epoch)", fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

print("Spearman ρ (val vs test):")
for col, label in METRICS:
    rho, _ = spearmanr(df[f"val_{col}"], df[f"test_{col}"])
    print(f"  {label:6s}  ρ={rho:.3f}")

In [ ]:
# Summary table: key epoch snapshots
snap_epochs = sorted({1, 100, 200, best_val_auprc_ep, 300, 500, best_test_auprc_ep, 700, 900})
rows = []
for ep_n in snap_epochs:
    row = df[df["epoch"] == ep_n]
    if row.empty:
        row = df.iloc[[ep_n - 1]]
    r = row.iloc[0]
    tag = ""
    if ep_n == best_val_auprc_ep:  tag += " ← best val AUPRC"
    if ep_n == best_test_auprc_ep: tag += " ← best test AUPRC (diag)"
    rows.append({
        "epoch": f"{ep_n}{tag}",
        "train_loss": f"{r['train_loss']:.3f}",
        "val AUROC":  f"{r['val_AUROC']:.3f}",
        "val AUPRC":  f"{r['val_AUPRC']:.3f}",
        "val MCC":    f"{r['val_MCC']:.3f}",
        "test AUROC": f"{r['test_AUROC']:.3f}",
        "test AUPRC": f"{r['test_AUPRC']:.3f}",
        "test MCC":   f"{r['test_MCC']:.3f}",
    })

display(pd.DataFrame(rows).set_index("epoch"))

## Выводы

- Val метрики **немонотонны**: есть провалы с последующим восстановлением — ранняя остановка по декодеру 
  на epoch 300 обрывает полезное дообучение энкодера.
- Best val AUPRC ≠ best epoch для downstream XGBoost probe: 900-эп энкодер даёт лучший probe 
  даже при val AUPRC ниже пика.
- Val ↔ Test Spearman ρ ~0.76–0.80: грубая корреляция есть, но точечный выбор эпохи по val AUPRC 
  не оптимален для probe.

**Рекомендация:** сохранять энкодер-чекпоинты (каждые 50–100 эпох), выбирать эпоху по 
**val probe AUPRC** (XGBoost на train/val), тест трогать только один раз.